In [1]:
#Setup & Library Imports

In [3]:
!pip install torch torchvision torchaudio captum shap lime ucimlrepo pandas numpy matplotlib scikit-learn

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/11

In [4]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip install captum shap lime ucimlrepo pandas numpy matplotlib scikit-learn

Looking in indexes: https://download.pytorch.org/whl/cpu


In [5]:
# Install required libraries (Run this once)
# Note: If you are using VS Code, you might need to run these in your terminal without the '!'
# !pip install -q captum shap lime ucimlrepo torch pandas numpy matplotlib scikit-learn

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from ucimlrepo import fetch_ucirepo

# Import XAI Tools
import shap
import lime
from lime import lime_tabular
from captum.attr import IntegratedGradients, DeepLift, LayerConductance

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("═" * 30)
print("Environment Setup Complete")
print("═" * 30)

Matplotlib is building the font cache; this may take a moment.


══════════════════════════════
Environment Setup Complete
══════════════════════════════


In [6]:
#Data Pipeline

In [7]:
#Data Pipeline: Fetching and Preprocessing
print("[1/6] Fetching Breast Cancer Wisconsin Diagnostic dataset (ID: 17)...")

# Fetching the official UCI dataset
repo = fetch_ucirepo(id=17) 
X_df = repo.data.features
y_series = repo.data.targets['Diagnosis']

# Mapping diagnosis: 'M' -> 0 (Malignant), 'B' -> 1 (Benign)
# This mapping is standard for the Breast Cancer Wisconsin dataset [cite: 111]
y_arr = (y_series == 'B').astype(int).values
X_arr = X_df.values.astype(np.float64)
feature_names = list(X_df.columns)

# Stratified split: 64% train / 16% val / 20% test
X_tv, X_test, y_tv, y_test = train_test_split(
    X_arr, y_arr, test_size=0.20, random_state=42, stratify=y_arr)
X_train, X_val, y_train, y_val = train_test_split(
    X_tv, y_tv, test_size=0.20, random_state=42, stratify=y_tv)

# Standardization: Scaling features for the DNN
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# Convert to PyTorch Tensors for processing
Xt = torch.tensor(X_train, dtype=torch.float32)
yt = torch.tensor(y_train, dtype=torch.long)
Xv = torch.tensor(X_val, dtype=torch.float32)
yv = torch.tensor(y_val, dtype=torch.long)
Xte = torch.tensor(X_test, dtype=torch.float32)
yte = torch.tensor(y_test, dtype=torch.long)

print(f"      ✓ Dataset Loaded: {X_arr.shape[0]} samples")
print(f"      ✓ Features Cleaned: {len(feature_names)} clinical markers")
print(f"      ✓ Data Split Complete: Train={len(Xt)}, Val={len(Xv)}, Test={len(Xte)}")

[1/6] Fetching Breast Cancer Wisconsin Diagnostic dataset (ID: 17)...


ConnectionError: Error connecting to server